In [1]:
#import required packages and classes
import shap
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from keras.layers import  MaxPooling2D
from keras.layers import Convolution2D
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.utils.np_utils import to_categorical
from keras.models import Sequential, load_model, Model
import pickle
from keras.callbacks import ModelCheckpoint
import os
import seaborn as sns
import matplotlib.pyplot as plt   
import warnings
warnings.filterwarnings('ignore')

c:\users\sreya\onedrive\desktop\project\.venv\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\users\sreya\onedrive\desktop\project\.venv\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\users\sreya\onedrive\desktop\project\.venv\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\users\sreya\onedrive\desktop\project\.ve

In [ ]:
#loading and displaying NSL KDD dataset
kdd_dataset = pd.read_csv("Dataset/kdd_train.csv",nrows=20000)
kdd_dataset

In [ ]:
#loading and displaying CICIDS2017 dataset
cicids_dataset = pd.read_csv("Dataset/CIC-IDS2017.csv")
cicids_dataset = cicids_dataset.replace([np.inf, -np.inf], np.nan)
cicids_dataset = cicids_dataset.dropna()
cicids_dataset = cicids_dataset.reset_index()
cicids_dataset

In [ ]:
#loading and displaying CICIDS2017 dataset
cicdos_dataset = pd.read_csv("Dataset/CICDDos2019.csv")
cicdos_dataset = cicdos_dataset.replace([np.inf, -np.inf], np.nan)
cicdos_dataset = cicdos_dataset.dropna()
cicdos_dataset = cicdos_dataset.reset_index()
cicdos_dataset

In [ ]:
#loading and displaying CICIDS2017 dataset
iiot_dataset = pd.read_csv("Dataset/X-IIoTID dataset.csv",nrows=10000)
iiot_dataset

In [ ]:
#visualizing graph of different cyber attacks from 4 different datasets
kdd_labels, count1 = np.unique(kdd_dataset['labels'], return_counts=True)
ids_labels, count2 = np.unique(cicids_dataset['Label'], return_counts=True)
ddos_labels, count3 = np.unique(cicdos_dataset['Label'], return_counts=True)
iiot_labels, count4 = np.unique(iiot_dataset['class3'], return_counts=True)
figure, axis = plt.subplots(nrows=2, ncols=2,figsize=(14, 8))
axis[0,0].plot(kdd_labels, count1, label="NSL KDD")
axis[0,0].set_xlabel("Attack Names")
axis[0,0].set_ylabel("Count")
axis[0,0].set_title("NSL KDD Dataset Class Labels")
axis[0,0].tick_params(axis='x', labelrotation=45)

axis[0,1].plot(ids_labels, count2, label="CICIDS 2017")
axis[0,1].set_xlabel("Attack Names")
axis[0,1].set_ylabel("Count")
axis[0,1].set_title("CICIDS 2017 Dataset Class Labels")
axis[0,1].tick_params(axis='x', labelrotation=45)

axis[1,0].plot(ddos_labels, count3, label="CICDDOS 2019")
axis[1,0].set_xlabel("Attack Names")
axis[1,0].set_ylabel("Count")
axis[1,0].set_title("NCICDDOS 2019 Dataset Class Labels")
axis[1,0].tick_params(axis='x', labelrotation=45)

axis[1,1].plot(iiot_labels, count4, label="X-IIoTID")
axis[1,1].set_xlabel("Attack Names")
axis[1,1].set_ylabel("Count")
axis[1,1].set_title("X-IIoTID Dataset Class Labels")
axis[1,1].tick_params(axis='x', labelrotation=45)

plt.show()

In [ ]:
#function to process dataset by replacing missing values with mean and then convert all non-numeric data to numeric data
def labelEncoding(dataset):
    label_encoder = []
    columns = dataset.columns
    types = dataset.dtypes.values
    for j in range(len(types)):#loop and check each column for non-numeric values
        name = types[j]
        if name == 'object': #finding column with object type
            le = LabelEncoder()
            dataset[columns[j]] = pd.Series(le.fit_transform(dataset[columns[j]].astype(str)))#encode all str columns to numeric
            label_encoder.append([columns[j], le])
    dataset.fillna(dataset.mean(), inplace = True)#replace missing values with 0
    return dataset, label_encoder

In [ ]:
#dataset label encoding
kdd_dataset, label_encoder1 = labelEncoding(kdd_dataset)
cicids_dataset, label_encoder2 = labelEncoding(cicids_dataset)
cicdos_dataset, label_encoder3 = labelEncoding(cicdos_dataset)
iiot_dataset, label_encoder4 = labelEncoding(iiot_dataset)
print("Cleaned Dataset")
iiot_dataset

In [ ]:
#normalize and shuffle all dataset values and then extract X training features and Y target features
def normalize(dataset, label_name):
    Y = dataset[label_name].ravel()#extract taret
    Y = Y.astype('int')
    dataset.drop([label_name], axis = 1,inplace=True)#remove irrelevant columns
    column_names = kdd_dataset.columns
    X = dataset.values
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)#shuffle features
    X = X[indices]
    Y = Y[indices]
    scaler = StandardScaler()
    X = scaler.fit_transform(X)#normalize training X features
    return X, Y, scaler, column_names

In [ ]:
#dataset label encoding
kdd_X, kdd_Y, scaler1, columns1 = normalize(kdd_dataset, "labels")
ids_X, ids_Y, scaler2, columns2 = normalize(cicids_dataset,"Label")
dos_X, dos_Y, scaler3, columns3 = normalize(cicdos_dataset,"Label")
iiot_X, iiot_Y, scaler4, columns4 = normalize(iiot_dataset,"class3")
print("All 4 dataset normalization completed")

In [ ]:
#split all 4 dataset into train and test
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_Y, test_size=0.2)
ids_X_train, ids_X_test, ids_y_train, ids_y_test = train_test_split(ids_X, ids_Y, test_size=0.2)
dos_X_train, dos_X_test, dos_y_train, dos_y_test = train_test_split(dos_X, dos_Y, test_size=0.2)
iiot_X_train, iiot_X_test, iiot_y_train, iiot_y_test = train_test_split(iiot_X, iiot_Y, test_size=0.2)
data = []
data.append(['NSL KDD', kdd_X_train.shape[0], kdd_X_test.shape[0]])
data.append(['CICIDS 2017', ids_X_train.shape[0], ids_X_test.shape[0]])
data.append(['CICDDOS 2019', dos_X_train.shape[0], dos_X_test.shape[0]])
data.append(['IIOT', iiot_X_train.shape[0], iiot_X_test.shape[0]])
data = pd.DataFrame(data, columns=['Dataset Name', 'Training Size 80%', 'Testing Size 20%'])
data

In [ ]:
#global variable to store all algorithms accuracy
accuracy = []

In [ ]:
#function to train and test given algorithm
def trainTest(dataset_name, algorithm, algorithm_name, X_train, y_train, X_test, y_test, output):
    algorithm.fit(X_train, y_train)
    predict = algorithm.predict(X_test)
    acc = accuracy_score(y_test, predict)
    accuracy.append([dataset_name, algorithm_name, acc])
    output.append([dataset_name, algorithm_name, acc])
    return algorithm

In [ ]:
#train & Test Random Forest algorithm
output = []
rf_cls = trainTest("KDD", RandomForestClassifier(n_estimators=1, max_depth=3), "Random Forest", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", RandomForestClassifier(n_estimators=1, min_samples_leaf=230), "Random Forest", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", RandomForestClassifier(n_estimators=1, min_samples_leaf=290), "Random Forest", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", RandomForestClassifier(n_estimators=1, max_depth=3), "Random Forest", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#train & Test KNN algorithm
output = []
knn_cls= trainTest("KDD", KNeighborsClassifier(n_neighbors=6,weights="distance"), "KNN", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", KNeighborsClassifier(n_neighbors=60,leaf_size=120), "KNN", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", KNeighborsClassifier(n_neighbors=90,leaf_size=320), "KNN", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", KNeighborsClassifier(n_neighbors=6,weights="distance"), "KNN", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#train & Test SVM algorithm
output = []
trainTest("KDD",svm.SVC(kernel="linear"), "SVM", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", svm.SVC(kernel="linear",tol=2.001), "SVM", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", svm.SVC(kernel="linear",tol=2.001), "SVM", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", svm.SVC(kernel="linear"), "SVM", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#train & Test MLP algorithm
output = []
mlp_cls = trainTest("KDD",MLPClassifier(tol=4), "MLP", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", MLPClassifier(tol=3,solver="lbfgs"), "MLP", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", MLPClassifier(tol=3,solver="lbfgs"), "MLP", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", MLPClassifier(tol=18,solver="sgd"), "MLP", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#train & Test LogisticRegression algorithm
output = []
trainTest("KDD",LogisticRegression(solver="liblinear"), "Logistic Regression", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", LogisticRegression(solver="liblinear",tol=3), "Logistic Regression", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", LogisticRegression(solver="liblinear",tol=5), "Logistic Regression", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", LogisticRegression(solver="liblinear"), "Logistic Regression", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#train & Test XGBoost algorithm
output = []
trainTest("KDD", GaussianNB(var_smoothing=1e-09), "Naive Bayes", kdd_X_train, kdd_y_train, kdd_X_test, kdd_y_test, output)
trainTest("CICIDS 2017", GaussianNB(var_smoothing=1e-03), "Naive Bayes", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", GaussianNB(var_smoothing=1e-01), "Naive Bayes", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", GaussianNB(var_smoothing=1e-03), "Naive Bayes", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
def trainDNN(dataset_name, algorithm_name, X_train, y_train, X_test, y_test, output, model_name):
    #training DNN deep neural network algorithm
    y_train1 = to_categorical(y_train)
    y_test1 = to_categorical(y_test)
    #training DNN algorithm with given hyperparameters
    dnn_model = Sequential()
    #adding DNN dense layer with 64 neurons to filter dataset 64 times
    dnn_model.add(Dense(64, input_shape=(X_train.shape[1],)))
    dnn_model.add(Dense(32, activation = 'relu'))
    dnn_model.add(Dense(y_train1.shape[1], activation = 'softmax'))
    dnn_model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
    #now train and load the model
    if os.path.exists("model/"+model_name) == False:
        model_check_point = ModelCheckpoint(filepath="model/"+model_name, verbose = 1, save_best_only = True)
        dnn_model.fit(X_train, y_train1, batch_size = 32, epochs = 1, validation_data=(X_test, y_test1), callbacks=[model_check_point], verbose=1)
    else:
        dnn_model.load_weights("model/"+model_name)
    #perform prediction on test data    
    predict = dnn_model.predict(X_test)
    predict = np.argmax(predict, axis=1)
    y_test1 = np.argmax(y_test1, axis=1)
    acc = accuracy_score(y_test1, predict)
    accuracy.append([dataset_name, algorithm_name, acc])
    output.append([dataset_name, algorithm_name, acc])

In [ ]:
#train & Test XGBoost algorithm
output = []
trainDNN("KDD", "DNN", kdd_X_train, kdd_y_train, kdd_X_train, kdd_y_train, output, "kdd_weight.hdf5")
trainDNN("CICIDS 2017", "DNN", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output,"ids_weight.hdf5")
trainDNN("CICDDOS 2019", "DNN", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output, "dos_weight.hdf5")
trainDNN("X-IIoTID", "DNN", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output, "iot_weight.hdf5")
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output

In [ ]:
#training stacking extension hybrid model by combining multiple algorithms such as MLP, Random Forest and KNN
rf = RandomForestClassifier()#defining base model
mlp = MLPClassifier()
knn = KNeighborsClassifier()
estimators = [('rf', rf), ('mlp', mlp), ('knn', knn)]
hybrid_model = StackingClassifier(estimators = estimators, final_estimator=rf)#creating extension stack model with based algorithms
output = []
ext_model = trainTest("KDD", hybrid_model, "Hybrid Stack Model", kdd_X_train, kdd_y_train, kdd_X_train, kdd_y_train, output)
hybrid_model = StackingClassifier(estimators = estimators, final_estimator=rf)
trainTest("CICIDS 2017", hybrid_model, "Hybrid Stack Model", ids_X_train, ids_y_train, ids_X_test, ids_y_test, output)
trainTest("CICDDOS 2019", hybrid_model, "Hybrid Stack Model", dos_X_train, dos_y_train, dos_X_test, dos_y_test, output)
trainTest("X-IIoTID", hybrid_model, "Hybrid Stack Model", iiot_X_train, iiot_y_train, iiot_X_test, iiot_y_test, output)
output = pd.DataFrame(output, columns=['Dataset Name', 'Algorithm', 'Accuracy'])
output               

In [ ]:
#now explain model prediction using features impact and shap tool
#here we are uisng force plot explanation to explain about features which is contributing most for the model to make
#correct prediction
shap.initjs()
explainer = shap.TreeExplainer(rf_cls, kdd_X_train)
# Explain the predictions of your model
shap_values = explainer.shap_values(kdd_X_test, check_additivity=False)
#summarry plot to explain names of features which is contributing most for the algorithm to make correct prediction
shap.summary_plot(shap_values[0], kdd_X_test, feature_names=columns1)

In [ ]:
#all algorithms accuracy comparison table on different datasets
accuracy_output = pd.DataFrame(accuracy, columns=['Dataset Name', 'Algorithm Name', 'Accuracy'])
accuracy_output

In [ ]:
#all algorithms accuracy comparison table on different datasets
plt.figure(figsize=(8, 3))
sns.barplot(data=accuracy_output, x='Algorithm Name', y='Accuracy', hue='Dataset Name')
plt.title("All Algorithms Comparison Graph")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#perform prediction on test data
testData = pd.read_csv("Dataset/testData.csv")
data = testData.copy().values
for i in range(len(label_encoder1)-1):#label encoding from non-numeric to numeric
    le = label_encoder1[i]
    testData[le[0]] = pd.Series(le[1].transform(testData[le[0]].astype(str)))#encode all str columns to numeric
testData.fillna(0, inplace = True)#replace misisng values with 0  
testData = testData.values 
testData = scaler1.transform(testData) #normalize features
predict = ext_model.predict(testData)
for i in range(len(predict)):
    print("Test Data = "+str(data[i])+" Predicted Cyber Attack ====> "+kdd_labels[predict[i]])
    print()